In [1]:
import pandas as pd
from dataset_class.job_post_dataset import JobPostingDataset
from sklearn.model_selection import train_test_split
import nltk
from nltk.tokenize import word_tokenize
from collections import Counter
import numpy as np
from gensim.models import FastText, Word2Vec
from itertools import product
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['PYTHONWARNINGS'] = 'ignore'


nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Chadrick\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Chadrick\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### Splitting 

This section serves to split the dataset into Train, validation and test sets

In [2]:
numeric_cols = ["telecommuting", "missing_count", "total_text_len", "company_profile_len", "description_len", 
                "requirements_len", "benefits_len", "company_profile_word_count", "description_word_count", 
                "requirements_word_count", "benefits_word_count", "salary_provided", "has_company_profile",
                "vague_location", "has_company_logo", "has_questions"]

In [3]:
### loading clean text
df = pd.read_csv("./data/clean/fake_job_postings_ALL.csv")
df.head()


,full_text,telecommuting,missing_count,total_text_len,company_profile_len,description_len,requirements_len,benefits_len,company_profile_word_count,description_word_count,requirements_word_count,benefits_word_count,salary_provided,has_company_profile,vague_location,has_company_logo,has_questions,fraudulent
0,marketing intern us ny new york we are food 52...,0,3,2642,885,905,852,0,141,124,115,0,0,1,0,1,0,0
1,customer service cloud video production nz auc...,0,1,6088,1286,2077,1433,1292,153,315,200,227,0,1,0,1,0,0
2,commissioning machinery assistant cma us ia we...,0,6,2597,879,355,1363,0,141,50,164,0,0,1,0,1,0,0
3,account executive washington dc us dc washingt...,0,0,5425,614,2600,1429,782,85,346,176,97,0,1,0,1,0,0
4,bill review manager us fl fort worth spot sour...,0,0,3926,1628,1520,757,21,207,168,89,3,0,1,0,1,1,0


In [4]:
# 1. Split 80% Train, 20% "Rest" (temp_data)
train_data, temp_data = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['fraudulent']
)

# 2. Split that 20% into half (10% Val, 10% Test)
# FIX: Use temp_data['fraudulent'] for stratification
val_data, test_data = train_test_split(
    temp_data, test_size=0.5, random_state=42, stratify=temp_data['fraudulent']
)

In [5]:
X_train_text = train_data['full_text']
X_train_numeric = train_data[numeric_cols]
y_train = train_data['fraudulent']

X_val_text = val_data['full_text']
X_val_numeric = val_data[numeric_cols]
y_val = val_data['fraudulent']

X_test_text = test_data['full_text']
X_test_numeric = test_data[numeric_cols]
y_test = test_data['fraudulent']

### Tokenization

This sections serves to tokenize free text into a sequence of integers

### Embeddings

This section serves to convert the token ids into a high-dimensional vector to capture the semantic and syntactic meaning of the tokens

We will try a pretrained CBow as well as using FastText from scratch and evaluate their performance. 

#### FastText (using skipgram)

In [6]:
sentences = [word_tokenize(text.lower()) for text in X_train_text]  

In [7]:
fasttext_model = FastText(vector_size=100, window=5, min_count=5, sg=1) #assuming we want a 100 word vector
fasttext_model.build_vocab(sentences)
fasttext_model.train(sentences, total_examples=fasttext_model.corpus_count, epochs=10)

(47996520, 63557210)

Some checks if we we manage to learn technical jargon and correlated words

In [8]:
fasttext_model_optimal = FastText(vector_size=100, window=3, min_count=2, epochs=10, negative=5, sg = 1) 
fasttext_model_optimal.build_vocab(sentences)
fasttext_model_optimal.train(sentences, total_examples=fasttext_model_optimal.corpus_count, epochs=10)

(48260635, 63557210)

In [9]:
## Save the trained optimal FastText models
fasttext_model_optimal.save("optimal_fasttext.bin")

In [10]:
print(fasttext_model.wv['saas']) #technical jargon
print(fasttext_model.wv.vectors.shape) 

[-0.3945279   0.3343548  -0.18935499 -0.36743602 -0.41733742 -0.07341037
  0.43090174  0.39227697 -0.2629976  -0.5868029  -0.07671025 -0.2615613
 -0.17478853 -0.0099213  -0.42720976 -0.21260916  0.46807194  0.5671041
 -0.6850908  -0.6088931  -0.04999322  0.09462629 -0.3265151  -0.2522414
  0.15600438  0.25356713  0.48744047  0.10376747 -0.08252607 -0.3609249
  0.29098424  0.15940702  0.1650361  -0.29101187  0.08423879  0.07256889
 -0.15386985 -0.19449802 -0.18294153  0.3902467  -0.29554257 -0.41187546
  0.5107656  -0.30599934  0.42880994  0.2461876  -0.16227655  0.02038464
  0.20349856  0.382848    0.8340018  -0.2992495  -0.32935664  0.23102647
 -0.15001458  0.37971887 -0.3056345  -0.40007913 -0.40461406  0.07193333
  0.02069611  0.03518104  0.41037896 -0.273162    0.0763237   0.11834264
  0.578044   -0.11838421  0.16161498  0.16178241 -0.14938933 -0.0100866
  0.3688012  -0.46682456 -0.6780782  -0.10666108  0.2806834  -0.12433102
 -0.65961593  0.14530593  0.5815254   0.2649617   0.0081

In [11]:
print(fasttext_model.wv['bingsu']) #checking for oov
print(fasttext_model.wv.vectors.shape) 

[ 0.05725753  0.08052812 -0.27505487 -0.03385637 -0.1499753  -0.2774264
 -0.04497116  0.04763991 -0.04265158 -0.08364629 -0.18956493 -0.06278156
 -0.47231737  0.09246854 -0.05686537 -0.01396002  0.17116886  0.0801628
 -0.03707937 -0.0304885  -0.38163194 -0.22884923 -0.1828565   0.26440105
  0.24768604 -0.25288242 -0.2781584  -0.40886107  0.28706455 -0.2237224
 -0.03226085  0.19183928 -0.11342221  0.2941984   0.21030661  0.49268755
 -0.01567615 -0.1160615   0.06762169  0.289818   -0.02417179 -0.25933945
  0.22223473 -0.27302524 -0.41857272 -0.3846004  -0.39612687  0.17585689
 -0.2719546   0.3542383   0.4454432  -0.00577535 -0.03461864  0.46322885
 -0.2632197   0.09928815  0.26641458  0.24998039 -0.33099005  0.24897602
 -0.18152957 -0.19328271  0.23651025 -0.4285033  -0.05887862  0.09158158
 -0.18204682 -0.15183003  0.22013839  0.1761718  -0.31303865  0.04434197
  0.08909501 -0.09296902  0.0732087  -0.29120928  0.3694417   0.0993178
 -0.12159966  0.15369336 -0.14525479 -0.05669351  0.219

In [12]:
print(fasttext_model.wv.similarity('software', 'engineer'))
print(fasttext_model.wv.similarity('skills', 'experience'))
print(fasttext_model.wv.most_similar('water', topn=10))

print(fasttext_model_optimal.wv.similarity('software', 'engineer'))
print(fasttext_model_optimal.wv.similarity('skills', 'experience'))
print(fasttext_model_optimal.wv.most_similar('water', topn=10))


0.43773028
0.5137437
[('groundwater', 0.7944436073303223), ('wastewater', 0.7612429261207581), ('watering', 0.7253201603889465), ('stormwater', 0.7098804712295532), ('deepwater', 0.6821101307868958), ('hydration', 0.6551971435546875), ('waters', 0.6445022821426392), ('cwcb', 0.6266473531723022), ('hydro', 0.6245256066322327), ('hydraulic', 0.6239174604415894)]
0.43897995
0.5301568
[('groundwater', 0.7882607579231262), ('wastewater', 0.7729810476303101), ('watering', 0.7443079352378845), ('stormwater', 0.7438472509384155), ('waters', 0.7296249866485596), ('deepwater', 0.7160226702690125), ('waterfalls', 0.6929528117179871), ('waterloo', 0.6815873980522156), ('underwater', 0.6692019104957581), ('waterbabies', 0.6511003971099854)]


#### CBOW

In [13]:

cbow_model = Word2Vec(vector_size=100, window=5, min_count=5, sg=0) #assuming we want a 100 word vector
cbow_model.build_vocab(sentences)
cbow_model.train(sentences, total_examples=cbow_model.corpus_count, epochs=10)

(47995328, 63557210)

In [14]:
print(cbow_model.wv.similarity('software', 'engineer'))
print(cbow_model.wv.similarity('skills', 'experience'))
cbow_model.wv.most_similar('research', topn=10)


0.022028837
0.36963063


[('competitor', 0.621932327747345),
 ('qualitative', 0.5842775702476501),
 ('trends', 0.5585657358169556),
 ('analysis', 0.5563173294067383),
 ('analyses', 0.5351504683494568),
 ('segmentation', 0.5337792038917542),
 ('benchmarking', 0.49511370062828064),
 ('researching', 0.46828922629356384),
 ('findings', 0.4576430022716522),
 ('quantitative', 0.45519140362739563)]

### Hyperparameter tuning

We are going to tune the `sliding window size`, `embedding vector size`,`epochs`, `negative sampling` to obtain the optimal custom embedding which would be easily plugged into our model

we will measure through extrinsic evaluation and intrinsic evaluation

In [15]:
#to check oov rate
def oov_rate(model, corpus):
    oov = sum(1 for w in corpus if w not in model.wv)
    return oov/len(corpus)

#check top 10 words are similar to each other
def nearest_neighbour(model, test_words, topn = 10):
    scores = []
    for word in test_words:
        try:
            neighbours = model.wv.most_similar(word, topn=topn)
            scores.append(np.mean([score for _, score in neighbours]))
        except KeyError:
            pass
    return np.mean(scores)

#check if 2 correlated and 2 uncorrelated words are similar
def analogy_score(model, test_cases):
    correct = 0
    for pos1, pos2, neg1, expected in test_cases:
        try:
            results = model.wv.most_similar(
                positive=[pos1, pos2], negative=[neg1], topn=5
            )
            predicted = [w for w, _ in results]
            if expected in predicted:
                correct += 1
        except KeyError:
            pass
    return correct / len(test_cases)

In [16]:
#DO NOT RUN THIS IT WILL TAKE MANY HOURS

param_grid = {
    'vector_size': [100, 200],
    'window':      [3, 5, 10],
    'min_count':   [2, 5],
    'epochs':      [10, 20],
    'negative':    [5, 10],
}

test_words = ['engineer', 'manager', 'python', 'healthcare', 'experience', 'salary']
job_analogies = [
    ('engineer', 'python', 'manager', 'java'),
    ('senior', 'engineer', 'junior', 'developer'),
    ('full_time', 'salary', 'part_time', 'hourly'),
    ('healthcare', 'nurse', 'finance', 'analyst'),
]
vocab = [word for sentence in sentences for word in sentence]

results = []

keys = list(param_grid.keys())
combos = list(product(*param_grid.values()))
print(f"Total combinations: {len(combos)} runs")

for combo in combos:
    params = dict(zip(keys, combo))
    model = FastText(**params, sg=1, min_n=3, max_n=6)


    model.build_vocab(sentences)
    model.train(sentences, total_examples=model.corpus_count, epochs=params['epochs'])

    coherence = nearest_neighbour(model, test_words)
    oov       = oov_rate(model, vocab)
    analogy   = analogy_score(model, job_analogies)

    results.append({
            **params,
            'coherence':  round(coherence, 4),
            'oov_rate':   round(oov, 4),
            'analogy':    round(analogy, 4),
        })
    print(f" {params} → coherence={coherence:.4f}, oov={oov:.4f}, analogy={analogy:.4f}")

results_df = pd.DataFrame(results)
results_df.to_csv('data/clean/embedding_tuning.csv', index=False)

Total combinations: 48 runs
 {'vector_size': 100, 'window': 3, 'min_count': 2, 'epochs': 10, 'negative': 5} → coherence=0.7873, oov=0.0000, analogy=0.0000
 {'vector_size': 100, 'window': 3, 'min_count': 2, 'epochs': 10, 'negative': 10} → coherence=0.7846, oov=0.0000, analogy=0.0000
 {'vector_size': 100, 'window': 3, 'min_count': 2, 'epochs': 20, 'negative': 5} → coherence=0.7631, oov=0.0000, analogy=0.2500
 {'vector_size': 100, 'window': 3, 'min_count': 2, 'epochs': 20, 'negative': 10} → coherence=0.7581, oov=0.0000, analogy=0.2500
 {'vector_size': 100, 'window': 3, 'min_count': 5, 'epochs': 10, 'negative': 5} → coherence=0.7139, oov=0.0000, analogy=0.5000
 {'vector_size': 100, 'window': 3, 'min_count': 5, 'epochs': 10, 'negative': 10} → coherence=0.7132, oov=0.0000, analogy=0.5000
 {'vector_size': 100, 'window': 3, 'min_count': 5, 'epochs': 20, 'negative': 5} → coherence=0.6958, oov=0.0000, analogy=0.5000
 {'vector_size': 100, 'window': 3, 'min_count': 5, 'epochs': 20, 'negative': 10}